# 04 · Validation analysis: leaderboard, errors, feature drivers
Everything here uses **validation only** (2018-07 to 2018-12). It ends by freezing the model configurations,
which is the last step before 2019 is touched (Gate G2).

In [ ]:
import sys, os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from src import evaluate as ev, metrics as M, models as mo
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10, "axes.titleweight": "bold"})
def savefig(fig, name):
    fig.tight_layout(); fig.savefig(ev.FIGS / f"{name}.png", dpi=150); plt.show()
print("results folder:", ev.RESULTS, "| smoke mode:", mo.SMOKE, "| threads:", mo.N_JOBS)

## 1. Validation leaderboard (all models, both horizons)

In [ ]:
NAMES = {n: mo.LABEL[n] for n in mo.MODELS}
boards = {}
for h in ev.HORIZONS:
    lb = ev.leaderboard("val", h)
    bb = lb.attrs["best_baseline"]
    va = ev.load_table(h, ["val"])
    bt = ev.baseline_table(va)
    lb = lb.rename(index=NAMES)
    lb["Skill"] = lb["Skill"].astype(float)
    boards[h] = lb
    print(f"\n=== Horizon {h} min   (best baseline on validation: {bb}, MAE {bt.loc[bb, 'MAE']:.2f}) ===")
    display(lb.drop(columns="n").round(3))
    lb.to_csv(ev.RESULTS / f"val_leaderboard_h{h}.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, h in zip(axes, ev.HORIZONS):
    lb = boards[h].sort_values("MAE", ascending=False)
    base = ev.baseline_table(ev.load_table(h, ["val"]))
    ax.barh(lb.index, lb["MAE"], color="#0072B2")
    for i, name in enumerate(base.index):
        ax.axvline(base.loc[name, "MAE"], color=["#D55E00", "#009E73", "#E69F00"][i], ls="--", label=name)
    ax.set_xlabel("validation MAE (vehicles per 15 min)"); ax.set_title(f"{h}-minute horizon"); ax.legend(fontsize=8)
savefig(fig, "val_leaderboard")

## 2. When does each model win? Error by hour of day
Best model vs the best baseline, weekdays only, validation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=False)
for ax, h in zip(axes, ev.HORIZONS):
    top = boards[h].index[0]; key = [k for k, v in NAMES.items() if v == top][0]
    d = ev.load_preds("val", key, h); d = d[d["is_weekend"] == 0]
    bb = ev.best_baseline(ev.load_table(h, ["val"]))
    d["base"] = d[ev.BASELINES[bb]] * d["sc"]
    hr = d["target_time"].dt.hour
    ax.plot(d.groupby(hr).apply(lambda x: (x["y_raw"] - x["pred"]).abs().mean()), color="#0072B2", label=top)
    ax.plot(d.groupby(hr).apply(lambda x: (x["y_raw"] - x["base"]).abs().mean()), color="#D55E00", ls="--", label=f"{bb} baseline")
    ax.set_xlabel("hour of day"); ax.set_ylabel("MAE"); ax.set_title(f"{h}-minute horizon, weekdays"); ax.legend()
savefig(fig, "val_error_by_hour")

## 3. Residual diagnostics of the best model
Left: predicted vs actual (should hug the diagonal). Right: error grows with flow level, and its distribution has a
heavy tail (a few large misses, e.g. incidents).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for r, h in enumerate(ev.HORIZONS):
    top = boards[h].index[0]; key = [k for k, v in NAMES.items() if v == top][0]
    d = ev.load_preds("val", key, h).sample(min(200_000, len(ev.load_table(h, ["val"]))), random_state=0)
    axes[r, 0].hexbin(d["y_raw"], d["pred"], gridsize=60, bins="log", cmap="viridis", mincnt=1)
    lim = [0, max(d["y_raw"].max(), d["pred"].max())]; axes[r, 0].plot(lim, lim, "w--", lw=1)
    axes[r, 0].set_xlabel("actual"); axes[r, 0].set_ylabel("predicted"); axes[r, 0].set_title(f"{top}, {h} min"); axes[r, 0].grid(False)
    bins = pd.qcut(d["y_raw"], 10, duplicates="drop")
    e = (d["y_raw"] - d["pred"]).abs().groupby(bins, observed=True)
    axes[r, 1].plot(range(len(e.mean())), e.mean().to_numpy(), "o-", color="#0072B2", label="mean |error|")
    axes[r, 1].plot(range(len(e.mean())), e.quantile(.95).to_numpy(), "o--", color="#D55E00", label="95th percentile")
    axes[r, 1].set_xlabel("flow decile (low to high)"); axes[r, 1].set_title("Error by flow level"); axes[r, 1].legend()
savefig(fig, "val_residuals")

## 4. Feature drivers (LightGBM) - what the "Feature Drivers" panel in the app shows
Two views on the validation split: LightGBM **gain** importance (how much each feature improved splits) and
**permutation** importance (how much validation error rises when a feature is shuffled; this is the more trustworthy
one when features are correlated). Both are also summed per feature group.

In [ ]:
import joblib
from src.features import ALL_FEATURES, FEATURE_GROUPS
gmap = {f: g for g, fs in FEATURE_GROUPS.items() for f in fs}
drv = {}
for h in ev.HORIZONS:
    mdl = joblib.load(ev.MODELS_DIR / f"lgbm_h{h}_trainonly.joblib")
    va = ev.load_table(h, ["val"]).sample(min(40_000, 10**9), random_state=0)
    Xv, yv, scv = mo.xy(va)
    base = mo.val_mae_raw(yv, mdl.predict(Xv), scv)
    rng = np.random.default_rng(0)
    perm = {}
    for j, f in enumerate(ALL_FEATURES):
        Xp = Xv.copy(); Xp[:, j] = rng.permutation(Xp[:, j])
        perm[f] = mo.val_mae_raw(yv, mdl.predict(Xp), scv) - base
    gain = pd.Series(mdl.booster_.feature_importance("gain"), index=ALL_FEATURES)
    drv[h] = pd.DataFrame({"gain_share": gain / gain.sum(), "perm_MAE_increase": pd.Series(perm), "group": pd.Series(gmap)})
    drv[h].to_csv(ev.RESULTS / f"feature_drivers_h{h}.csv")
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for r, h in enumerate(ev.HORIZONS):
    d = drv[h].sort_values("perm_MAE_increase").tail(12)
    axes[r, 0].barh(d.index, d["perm_MAE_increase"], color="#0072B2"); axes[r, 0].set_title(f"Permutation importance, {h} min (top 12)")
    axes[r, 0].set_xlabel("increase in validation MAE when shuffled")
    g = drv[h].groupby("group")["perm_MAE_increase"].sum().sort_values()
    axes[r, 1].barh(g.index, g.values, color="#E69F00"); axes[r, 1].set_title(f"By feature group, {h} min")
savefig(fig, "val_feature_drivers")

## 5. Freeze the configuration (Gate G2, step 1)
Writes `results/frozen_config.json`: model, horizon, best parameters and the tree count that the final refit will
reuse. After this file exists, nothing about the models is changed any more; only then do you run the refit
(`python scripts/run_model.py refit --models all --h 15 60 --seeds`) and open notebook 05.

In [ ]:
frozen, missing = {}, []
for name in mo.MODELS:
    for h in ev.HORIZONS:
        r = ev.read_best(name, h)
        if r is None:
            missing.append(f"{name} h={h}"); continue
        frozen[f"{name}_h{h}"] = {"params": r["params"], "best_iter": r["best_iter"], "val_MAE": r["val"]["MAE"]}
print("Missing (run them first):", missing or "none")
if not missing:
    (ev.RESULTS / "frozen_config.json").write_text(json.dumps(frozen, indent=2))
    print("Frozen:", len(frozen), "configurations -> results/frozen_config.json")